In [1]:
import networkx as nx
import random
import numpy as np
import itertools
import re
from collections import Counter
import seaborn as sns
import pandas as pd
import pickle 
import matplotlib.pyplot as plt

In [3]:
with open('data/processed/all_c_elegans_weighted_graphs.pkl', 'rb') as file:
    all_c_elegans_weighted_graphs = pickle.load(file)
    
stage_labels = ['L1-0', 'L1-5', 'L1-8', 'L1-16', 'L2', 'L3', 'Adult-45-1', 'Adult-45-2']

with open('data/processed/C_elegans_kin_k_cores_across_stages.pkl', 'rb') as file:
    kin_k_cores_across_stages = pickle.load(file)
    
with open('data/processed/C_elegans_kout_k_cores_across_stages.pkl', 'rb') as file:
    kout_k_cores_across_stages = pickle.load(file)

with open('data/processed/C_elegans_kin_s_cores_across_stages.pkl', 'rb') as file:
    kin_s_cores_across_stages = pickle.load(file)
    
with open('data/processed/C_elegans_kout_s_cores_across_stages.pkl', 'rb') as file:
    kout_s_cores_across_stages = pickle.load(file)


In [5]:
with open('data/processed/C_elegans_lineage_distance_for_witvliet_data.pkl', 'rb') as file:
    cell_lineages_distances_of_witvliet_data = pickle.load(file)

In [6]:
cell_lineages_distances_of_witvliet_data

{('ADAR', 'AVAR'): 18,
 ('ADAR', 'BWM-VL01'): 19,
 ('ADAR', 'AVFL'): 19,
 ('ADAR', 'RMGL'): 16,
 ('ADAR', 'ADLR'): 12,
 ('ADAR', 'RMEL'): 18,
 ('ADAR', 'BWM-DR08'): 18,
 ('ADAR', 'BWM-DL08'): 18,
 ('ADAR', 'RIAL'): 18,
 ('ADAR', 'AIBL'): 16,
 ('ADAR', 'AFDL'): 18,
 ('ADAR', 'ASEL'): 19,
 ('ADAR', 'AVBL'): 16,
 ('ADAR', 'RIH'): 14,
 ('ADAR', 'PVR'): 18,
 ('ADAR', 'AIZL'): 17,
 ('ADAR', 'RMER'): 18,
 ('ADAR', 'SMDVR'): 18,
 ('ADAR', 'AINL'): 18,
 ('ADAR', 'RIGR'): 14,
 ('ADAR', 'SIAVL'): 16,
 ('ADAR', 'GLRDL'): 20,
 ('ADAR', 'AVM'): 11,
 ('ADAR', 'AQR'): 10,
 ('ADAR', 'RMDR'): 12,
 ('ADAR', 'CEPVR'): 15,
 ('ADAR', 'BWM-DL05'): 19,
 ('ADAR', 'RIPR'): 18,
 ('ADAR', 'URYDL'): 18,
 ('ADAR', 'CEPshVL'): 16,
 ('ADAR', 'OLLL'): 19,
 ('ADAR', 'AVDL'): 18,
 ('ADAR', 'PVQR'): 10,
 ('ADAR', 'BWM-VL08'): 18,
 ('ADAR', 'OLQDR'): 19,
 ('ADAR', 'SAAVR'): 14,
 ('ADAR', 'ASJR'): 13,
 ('ADAR', 'ALNL'): 18,
 ('ADAR', 'SAAVL'): 16,
 ('ADAR', 'ASGL'): 16,
 ('ADAR', 'DVC'): 18,
 ('ADAR', 'ADFR'): 12,
 ('ADAR'

In [7]:
def lineage_distance_of_group_of_neurons(iterable, graph, dict_of_cell_lineages):
    
    spclasses = set(graph.nodes[node]['specific_class'] for node in iterable)
    all_pairs = it.combinations(spclasses, 2)
    lineage_distances = [dict_of_cell_lineages[pair] for pair in all_pairs]
    
    return lineage_distances

In [8]:
import itertools as it

list(it.combinations(['A', 'B', 'C'], 2))

[('A', 'B'), ('A', 'C'), ('B', 'C')]

In [9]:
data_by_stage = {stage: {} for stage in stage_labels}

core_dicts = {
    "In k-core": kin_k_cores_across_stages,
    "Out k-core": kout_k_cores_across_stages,
    "In s-core": kin_s_cores_across_stages,
    "Out s-core": kout_s_cores_across_stages
}

for idx, graph in enumerate(all_c_elegans_weighted_graphs):
    all_nodes = set(graph.nodes())
    stage = stage_labels[idx]

    for core_label, core_dict in core_dicts.items():
        core_nodes = core_dict[idx]
        periphery_nodes = all_nodes - core_nodes

        core_dist = lineage_distance_of_group_of_neurons(core_nodes, graph, cell_lineages_distances_of_witvliet_data)
        periphery_dist = lineage_distance_of_group_of_neurons(periphery_nodes, graph, cell_lineages_distances_of_witvliet_data)
        data_by_stage[stage][f"{core_label} (Core)"] = core_dist
        data_by_stage[stage][f"{core_label} (Periphery)"] = periphery_dist

In [10]:
# Core types and their colors
core_types = ["In k-core", "Out k-core", "In s-core", "Out s-core"]
core_colors = {
    "In k-core": "cornflowerblue",
    "Out k-core": "orange",
    "In s-core": "green",
    "Out s-core": "crimson"
}

fig, axes = plt.subplots(2, 4, figsize=(16, 8), dpi=1000, sharey=True)
axes = axes.flatten()

for i, stage in enumerate(data_by_stage.keys()):
    ax = axes[i]
    labels = []
    data = []
    colors = []

    for core_type in core_types:
        core_label = f"{core_type} (Core)"
        peri_label = f"{core_type} (Periphery)"
        xcore_label = 'C'
        xperi_label = 'P'
        
        core_data = data_by_stage[stage][core_label]
        peri_data = data_by_stage[stage][peri_label]

        labels.extend([xcore_label, xperi_label])
        data.extend([core_data, peri_data])
        colors.extend([core_colors[core_type], core_colors[core_type]])

    # Create boxplot
    bp = ax.boxplot(data, patch_artist=True, medianprops=dict(linestyle='-', linewidth=1.5, color='lavender'), showmeans=True, 
                    meanprops=dict(marker='o', markerfacecolor='black', markeredgecolor='black',
                               markersize=5))
    
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)

    # Adding sample size annotations above each box
    for j, values in enumerate(data):
        n = len(values)
        max_val = max(values) if len(values) > 0 else 0
        ax.annotate(f"n={n}", xy=(j + 1, max_val), xytext=(0, 5), textcoords="offset points",
                    ha='center', va='bottom', fontsize=6, color='black')

    # multilevel x labeling
    sec = ax.secondary_xaxis(location=0)
    sec.set_xticks([1.5, 3.5, 5.5, 7.5], labels=['\n\n$k_{in}$', '\n\n$k_{out}$', '\n\n$s_{in}$', '\n\n$s_{out}$'], fontsize=15)
    sec.tick_params('x', length=0)

    sec2 = ax.secondary_xaxis(location=0)
    sec2.set_xticks([2.5, 4.5, 6.5], labels=[])
    sec2.tick_params('x', length=50, width=1)
    ax.set_xlim(0, 8.6)
    
    ax.set_title(stage, fontsize=14)
    ax.set_xticks(range(1, len(labels) + 1))
    ax.set_xticklabels(labels, fontsize=12)
    ax.set_ylabel("Lineage Distance")

    y_max = max([max(vals) for vals in data if len(vals) > 0])
    ax.set_ylim(top=y_max * 1.1)  # Adds 10% headroom to accomodate the sample size annotation

plt.tight_layout()
plt.show()


## Statistically comparing the distributions

In [31]:
# Checking the sample sizes 
for stage in stage_labels:
    print(f"\nStage: {stage}")
    for core_label in ["In k-core", "Out k-core", "In s-core", "Out s-core"]:
        core_key = f"{core_label} (Core)"
        peri_key = f"{core_label} (Periphery)"
        
        core_size = len(data_by_stage[stage][core_key])
        peri_size = len(data_by_stage[stage][peri_key])
        
        print(f"  {core_label}: Core (n={core_size}), Periphery (n={peri_size})")



Stage: L1-0
  In k-core: Core (n=105), Periphery (n=14706)
  Out k-core: Core (n=2080), Periphery (n=7381)
  In s-core: Core (n=28), Periphery (n=15931)
  Out s-core: Core (n=153), Periphery (n=14196)

Stage: L1-5
  In k-core: Core (n=105), Periphery (n=15753)
  Out k-core: Core (n=3081), Periphery (n=6441)
  In s-core: Core (n=45), Periphery (n=16653)
  Out s-core: Core (n=276), Periphery (n=14196)

Stage: L1-8
  In k-core: Core (n=1081), Periphery (n=11325)
  Out k-core: Core (n=3486), Periphery (n=6441)
  In s-core: Core (n=28), Periphery (n=17955)
  Out s-core: Core (n=6), Periphery (n=18721)

Stage: L1-16
  In k-core: Core (n=136), Periphery (n=17205)
  Out k-core: Core (n=3655), Periphery (n=6786)
  In s-core: Core (n=28), Periphery (n=18915)
  Out s-core: Core (n=6), Periphery (n=19701)

Stage: L2
  In k-core: Core (n=8001), Periphery (n=3403)
  Out k-core: Core (n=2346), Periphery (n=9870)
  In s-core: Core (n=28), Periphery (n=20301)
  Out s-core: Core (n=6), Periphery (n=211

**Running Mann-Whitney U test to see if the core and periphery distance distributions differ**

In [41]:
from scipy.stats import mannwhitneyu

p_values = {}

for stage in stage_labels:
    p_values[stage] = {}
    for core_label in ["In k-core", "Out k-core", "In s-core", "Out s-core"]:
        core_key = f"{core_label} (Core)"
        peri_key = f"{core_label} (Periphery)"
        
        core_vals = data_by_stage[stage][core_key]
        peri_vals = data_by_stage[stage][peri_key]

        stat, p = mannwhitneyu(core_vals, peri_vals, alternative='two-sided')
        p_values[stage][core_label] = p


In [29]:
p_values

{'L1-0': {'In k-core': 0.15572303335040688,
  'Out k-core': 5.139580949895415e-65,
  'In s-core': 7.671009343260894e-06,
  'Out s-core': 8.906717191434256e-08},
 'L1-5': {'In k-core': 0.005032533637367517,
  'Out k-core': 5.053686757479551e-150,
  'In s-core': 1.799409620175962e-07,
  'Out s-core': 1.9527850864496716e-23},
 'L1-8': {'In k-core': 0.041348649833837126,
  'Out k-core': 2.8672462214130195e-143,
  'In s-core': 3.450958694878765e-06,
  'Out s-core': 0.08570624438775097},
 'L1-16': {'In k-core': 0.055276823943099365,
  'Out k-core': 1.9967804377236195e-174,
  'In s-core': 2.5052985477184875e-06,
  'Out s-core': 0.07791325696743369},
 'L2': {'In k-core': 1.686014569472503e-12,
  'Out k-core': 9.504109160884385e-141,
  'In s-core': 1.343294045562911e-06,
  'Out s-core': 0.06895959755501363},
 'L3': {'In k-core': 1.1024167452513415e-06,
  'Out k-core': 7.655769151300401e-110,
  'In s-core': 8.982174624813407e-07,
  'Out s-core': 0.30850049021177106},
 'Adult-45-1': {'In k-core':

In [37]:
for stage in p_values:
    for coretype, pval in p_values[stage].items():
        if pval > 0.05:
            print(stage, '--->', coretype, pval)

L1-0 ---> In k-core 0.15572303335040688
L1-8 ---> Out s-core 0.08570624438775097
L1-16 ---> In k-core 0.055276823943099365
L1-16 ---> Out s-core 0.07791325696743369
L2 ---> Out s-core 0.06895959755501363
L3 ---> Out s-core 0.30850049021177106
Adult-45-2 ---> Out s-core 0.12857857599649095


In [77]:
a = ['ad', 'adf']
a.insert(0, 'dfdf')

**Running Mann-Whitney U test to see if the core neurons' lineage distances are lesser i.e. they are closely related**

_Including rank biserial correlation to check the practical significance of test result_

In [119]:
results_less = {}

for stage in stage_labels:
    results_less[stage] = {}
    for core_label in ["In k-core", "Out k-core", "In s-core", "Out s-core"]:
        core_key = f"{core_label} (Core)"
        peri_key = f"{core_label} (Periphery)"

        core_vals = data_by_stage[stage][core_key]
        peri_vals = data_by_stage[stage][peri_key]
        
        n1, n2 = len(core_vals), len(peri_vals)
        
        # Skip if either group is too small
        if n1 < 10 or n2 < 10:
            results_less[stage][core_label] = {"p": None, "r_rb": None, "note": "Too small"}
            continue

        stat, p = mannwhitneyu(core_vals, peri_vals, alternative='less')
        r_rb = 1 - (2 * stat) / (n1 * n2)

        results_less[stage][core_label] = {"p": p, "r_rb": r_rb}


In [121]:
for stage in results_less:
    print(f"\nStage: {stage}")
    for core_label in results_less[stage]:
        result = results_less[stage][core_label]
        
        if result["p"] is None:
            print(f"  {core_label}: Skipped (sample size too small)")
        else:
            p_val = result["p"]
            r_rb = result["r_rb"]
            print(f"  {core_label}: p = {p_val:.4e}, r_rb = {r_rb:.3f}")



Stage: L1-0
  In k-core: p = 7.7862e-02, r_rb = 0.079
  Out k-core: p = 2.5698e-65, r_rb = 0.240
  In s-core: p = 3.8355e-06, r_rb = 0.481
  Out s-core: p = 4.4534e-08, r_rb = 0.247

Stage: L1-5
  In k-core: p = 2.5163e-03, r_rb = 0.156
  Out k-core: p = 2.5268e-150, r_rb = 0.325
  In s-core: p = 8.9970e-08, r_rb = 0.442
  Out s-core: p = 9.7639e-24, r_rb = 0.344

Stage: L1-8
  In k-core: p = 9.7933e-01, r_rb = -0.037
  Out k-core: p = 1.4336e-143, r_rb = 0.304
  In s-core: p = 1.7255e-06, r_rb = 0.498
  Out s-core: Skipped (sample size too small)

Stage: L1-16
  In k-core: p = 2.7638e-02, r_rb = 0.094
  Out k-core: p = 9.9839e-175, r_rb = 0.329
  In s-core: p = 1.2526e-06, r_rb = 0.506
  Out s-core: Skipped (sample size too small)

Stage: L2
  In k-core: p = 8.4301e-13, r_rb = 0.082
  Out k-core: p = 4.7521e-141, r_rb = 0.330
  In s-core: p = 6.7165e-07, r_rb = 0.519
  Out s-core: Skipped (sample size too small)

Stage: L3
  In k-core: p = 5.5121e-07, r_rb = 0.064
  Out k-core: p = 3